# Intermediate 08 — Sampling Distributions: chi-square, t, and F

Practice notebook for the **"Sampling Distributions: chi-square, t, and F"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
# Part 1 — Chi-Square: Sum of Squared Standard Normals

If \(Z_1, \dots, Z_k\) are independent \(N(0,1)\) random variables, then

$$Q = \sum_{i=1}^{k} Z_i^2$$

has a **chi-square** distribution with \(k\) degrees of freedom, written \(Q \sim \chi_k^2\). Chi-square distributions arise naturally as sums of squared standardized residuals and as sampling distributions of variance estimators under normality.

Below we draw many standard-normal vectors of length \(k\), square and sum them, and overlay the theoretical \(\chi_k^2\) density from `scipy.stats.chi2`.


In [ ]:
# Simulate Q = sum of k squared standard normals, compare to chi2(k)
rng = np.random.default_rng(42)
k = 5
n_sim = 100_000

Z = rng.standard_normal(size=(n_sim, k))        # each row: k independent N(0,1)
Q = (Z**2).sum(axis=1)                            # chi-square(k) draws

# Theoretical density
q_grid = np.linspace(Q.min(), Q.max() * 0.95, 500)
pdf_theory = stats.chi2.pdf(q_grid, df=k)

fig, ax = plt.subplots()
ax.hist(Q, bins=80, density=True, color="tab:steelblue", alpha=0.7, label=rf"simulated $Q$, $k={k}$")
ax.plot(q_grid, pdf_theory, color="tab:red", lw=2, label=rf"$\chi^2_{{{k}}}$ PDF")
ax.set_xlabel(r"$Q = \sum_{i=1}^{k} Z_i^2$")
ax.set_ylabel("density")
ax.set_title("Sum of squared standard normals is chi-square")
ax.legend()
plt.show()

# Numerical check: mean and variance of chi-square(k) are k and 2k
print(f"empirical mean   = {Q.mean():.4f}   (theory {k})")
print(f"empirical var    = {Q.var():.4f}   (theory {2*k})")


**Your turn:** Change `k` to 1, 2, 10, 30 and re-run. How does the shape of the chi-square distribution change as the degrees of freedom grow? At large \(k\), does the distribution start to look symmetric and bell-shaped (a hint of the CLT)?


---
# Part 2 — Student's t from a Normal Sample

If \(Z \sim N(0,1)\) and \(Q \sim \chi_\nu^2\) are independent, then

$$T = \frac{Z}{\sqrt{Q/\nu}}$$

has a **Student's t** distribution with \(\nu\) degrees of freedom. In practice \(T\) appears when we replace the population standard deviation by the sample standard deviation in a normal model. For \(X_1, \dots, X_n \sim N(\mu, \sigma^2)\),

$$T = \frac{\bar{X} - \mu}{S/\sqrt{n}} \sim t_{n-1}.$$

We construct the t-statistic directly from normal samples (with \(\mu=0, \sigma=1\)) and compare its empirical distribution to the \(t_{n-1}\) PDF. Notice the heavier tails vs. the standard normal — that is the price of estimating \(\sigma\) from the same sample.


In [ ]:
# Construct the t-statistic from normal samples and compare to t_{n-1}
rng = np.random.default_rng(7)
n = 8                  # sample size -> df = n - 1 = 7
n_sim = 100_000
mu, sigma = 0.0, 1.0

samples = rng.normal(loc=mu, scale=sigma, size=(n_sim, n))
xbar = samples.mean(axis=1)
S = samples.std(axis=1, ddof=1)                 # sample std (denominator n-1)
T_stat = (xbar - mu) / (S / np.sqrt(n))          # t-statistic with df = n-1

df_t = n - 1
t_grid = np.linspace(-6, 6, 500)
pdf_t = stats.t.pdf(t_grid, df=df_t)
pdf_norm = stats.norm.pdf(t_grid)                # N(0,1) for comparison

fig, ax = plt.subplots()
ax.hist(T_stat, bins=120, density=True, color="tab:steelblue", alpha=0.6, label=rf"simulated $T$, $n={n}$")
ax.plot(t_grid, pdf_t, color="tab:red", lw=2, label=rf"$t_{{{df_t}}}$ PDF")
ax.plot(t_grid, pdf_norm, color="black", ls="--", lw=1.5, label=r"$N(0,1)$")
ax.set_xlabel(r"$T = (\bar{X}-\mu)/(S/\sqrt{n})$")
ax.set_ylabel("density")
ax.set_title("The t-statistic follows a Student's t distribution")
ax.legend()
plt.show()

# Tail check: t has heavier tails than normal. P(|T| > 2)
emp_t = np.mean(np.abs(T_stat) > 2)
print(f"P(|T| > 2) empirical = {emp_t:.4f}")
print(f"P(|t_{{n-1}}| > 2) theory = {2*stats.t.sf(2, df=df_t):.4f}")
print(f"P(|Z|    > 2) theory = {2*stats.norm.sf(2):.4f}  (lighter tails)")


**Your turn:** Increase the sample size `n` to 30 and then to 100. How does the gap between the \(t_{n-1}\) density and the \(N(0,1)\) density change? Why does the t distribution converge to the standard normal as \(n \to \infty\)?


---
# Part 3 — F Distribution: Ratio of Independent Chi-Squares

If \(Q_1 \sim \chi_{d_1}^2\) and \(Q_2 \sim \chi_{d_2}^2\) are independent, then

$$F = \frac{Q_1 / d_1}{Q_2 / d_2}$$

has an **F distribution** with \((d_1, d_2)\) degrees of freedom. F distributions appear as ratios of variance estimates and are central to analysis of variance (ANOVA) and multiple regression F-tests.

We build \(Q_1\) and \(Q_2\) by summing squared standard normals with \(d_1\) and \(d_2\) degrees of freedom respectively, form the ratio, and overlay the theoretical `scipy.stats.f` density.


In [ ]:
# Build F = (Q1/d1) / (Q2/d2) from independent chi-squares and compare to theory
rng = np.random.default_rng(123)
d1, d2 = 5, 10
n_sim = 100_000

# Independent chi-square draws via sums of squared standard normals
Z1 = rng.standard_normal(size=(n_sim, d1))
Z2 = rng.standard_normal(size=(n_sim, d2))
Q1 = (Z1**2).sum(axis=1)                          # chi2(d1)
Q2 = (Z2**2).sum(axis=1)                          # chi2(d2), independent of Q1
F_stat = (Q1 / d1) / (Q2 / d2)                   # F(d1, d2)

# Theoretical F density
f_grid = np.linspace(1e-3, F_stat.max() * 0.9, 500)
pdf_f = stats.f.pdf(f_grid, dfn=d1, dfd=d2)

fig, ax = plt.subplots()
ax.hist(F_stat, bins=120, density=True, color="tab:steelblue", alpha=0.6,
        label=rf"simulated $F$, $d_1={d1}, d_2={d2}$")
ax.plot(f_grid, pdf_f, color="tab:red", lw=2, label=rf"$F_{{{d1},{d2}}}$ PDF")
ax.set_xlabel(r"$F = (Q_1/d_1)/(Q_2/d_2)$")
ax.set_ylabel("density")
ax.set_title("Ratio of independent chi-squares is F-distributed")
ax.legend()
plt.show()

# Numerical check: mean of F(d1, d2) is d2 / (d2 - 2) for d2 > 2
print(f"empirical mean = {F_stat.mean():.4f}   (theory {d2/(d2-2):.4f})")
print(f"empirical var  = {F_stat.var():.4f}   "
      f"(theory {2*d2**2*(d1+d2-2)/(d1*(d2-2)**2*(d2-4)):.4f})")


**Your turn:** Swap \(d_1\) and \(d_2\) (set `d1, d2 = 10, 5`). How does the shape of the F distribution change? Also try `d1, d2 = 30, 30` — does the distribution become more concentrated around 1? Explain why \(E[F] \to 1\) as both degrees of freedom grow.


---
# Part 4 — The Three Sampling Distributions Side by Side

The three distributions in this section are built from the same ingredient — squared (and possibly normalized) standard normals. Here we overlay all three densities on a single panel for the same degrees of freedom \(\nu = 10\) so you can compare their shapes, supports, and tail behavior at a glance.

- \(\chi_\nu^2\): nonnegative, right-skewed, mean \(\nu\).
- \(t_\nu\): symmetric around 0, heavier tails than \(N(0,1)\), variance \(\nu/(\nu-2)\).
- \(F_{\nu, \nu}\): positive, mean \(\nu/(\nu-2)\), right tail heavier than chi-square.


In [ ]:
nu = 10
# chi-square(nu): support [0, inf)
x_pos = np.linspace(1e-3, 40, 500)
chi_pdf = stats.chi2.pdf(x_pos, df=nu)

# t(nu): support (-inf, inf)
x_all = np.linspace(-6, 6, 500)
t_pdf = stats.t.pdf(x_all, df=nu)

# F(nu, nu): support (0, inf), rescale to compare on same positive axis
f_pdf = stats.f.pdf(x_pos, dfn=nu, dfd=nu)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Left: positive-support densities (chi-square and F)
axes[0].plot(x_pos, chi_pdf, color="tab:blue", lw=2, label=rf"$\chi^2_{{{nu}}}$")
axes[0].plot(x_pos, f_pdf, color="tab:green", lw=2, label=rf"$F_{{{nu},{nu}}}$")
axes[0].set_xlim(0, 40); axes[0].set_xlabel("value"); axes[0].set_ylabel("density")
axes[0].set_title("Positive-support sampling distributions")
axes[0].legend()

# Right: t vs. standard normal (symmetric densities)
axes[1].plot(x_all, t_pdf, color="tab:red", lw=2, label=rf"$t_{{{nu}}}$")
axes[1].plot(x_all, stats.norm.pdf(x_all), color="black", ls="--", lw=1.5, label=r"$N(0,1)$")
axes[1].set_xlabel("value"); axes[1].set_ylabel("density")
axes[1].set_title(r"Student's $t$ vs. standard normal")
axes[1].legend()

fig.suptitle(r"Sampling distributions: $\chi^2$, $t$, and $F$ (all with $\nu=10$)", y=1.03)
plt.tight_layout(); plt.show()

# Variance comparison: t_10 has variance nu/(nu-2), heavier than 1
print(f"Var[t_{{10}}] = {nu/(nu-2):.4f}   vs.  Var[N(0,1)] = 1.0000")
print(f"Mean[F_{{10,10}}] = {nu/(nu-2):.4f}   vs.  Mean[chi^2_{{10}}] = {nu:.4f}")


**Your turn:** Set \(\nu = 3\). What happens to the variance of \(t_\nu\)? Why is it said that the t distribution has 'fat tails' for small \(\nu\), and why does this matter for inference with small samples?


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. State the definition of the **chi-square distribution**. If \(Z_1, \dots, Z_k\) are independent \(N(0,1)\), what is the distribution of \(\sum_{i=1}^k Z_i^2\)? Give its mean and variance.

2. Simulate \(Q = \sum_{i=1}^{k} Z_i^2\) for \(k = 4\) using 100,000 replications with a fixed seed. Overlay the histogram with the theoretical \(\chi^2_4\) density and report the empirical mean and variance. Confirm they match \(k\) and \(2k\).

3. Let \(X_1, \dots, X_n \sim N(\mu, \sigma^2)\) with \(n = 6\). Simulate 100,000 samples (use \(\mu = 2, \sigma = 3\)) and compute \(T = (\bar{X} - \mu)/(S/\sqrt{n})\) for each. Overlay the histogram of \(T\) with the \(t_{n-1}\) and \(N(0,1)\) densities. Report the empirical \(P(|T| > 2)\) and compare it to both theoretical values.

4. Construct an \(F_{4, 12}\) random variable as \(F = (Q_1/4)/(Q_2/12)\) where \(Q_1, Q_2\) are independent chi-squares obtained from sums of squared standard normals. Simulate 100,000 draws with a fixed seed, overlay the \(F_{4,12}\) PDF, and report the empirical mean against the theoretical \(d_2/(d_2-2)\).

5. Explain in 2-3 sentences why the t distribution has heavier tails than the standard normal, and why \(t_\nu \to N(0,1)\) as \(\nu \to \infty\). Support your answer numerically by computing \(P(|T| > 3)\) for \(t_\nu\) with \(\nu = 3, 10, 50, 1000\) and comparing to the normal value.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** If \(Z_1, \dots, Z_k \stackrel{\text{iid}}{\sim} N(0,1)\), then \(Q = \sum_{i=1}^{k} Z_i^2 \sim \chi_k^2\). Its mean is \(E[Q] = k\) and its variance is \(\operatorname{Var}(Q) = 2k\). Each \(Z_i^2 \sim \chi^2_1\), and independent chi-squares add, so the sum has \(k\) degrees of freedom.

**2.**

```python
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
k = 4
Z = rng.standard_normal(size=(100_000, k))
Q = (Z**2).sum(axis=1)
xs = np.linspace(0, Q.max()*0.95, 400)
plt.hist(Q, bins=80, density=True, alpha=0.7)
plt.plot(xs, stats.chi2.pdf(xs, df=k), "r-", lw=2)
plt.show()
print(f"mean = {Q.mean():.4f} (theory {k}), var = {Q.var():.4f} (theory {2*k})")
```
The empirical mean/variance should match 4 and 8 closely.

**3.** With \(n=6\), the t-statistic has \(n-1 = 5\) degrees of freedom.

```python
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
rng = np.random.default_rng(1)
n = 6
mu, sigma = 2.0, 3.0
samples = rng.normal(loc=mu, scale=sigma, size=(100_000, n))
xbar = samples.mean(axis=1)
S = samples.std(axis=1, ddof=1)
T = (xbar - mu) / (S / np.sqrt(n))
xs = np.linspace(-6, 6, 400)
plt.hist(T, bins=120, density=True, alpha=0.6)
plt.plot(xs, stats.t.pdf(xs, df=n-1), "r-", lw=2, label="t_5")
plt.plot(xs, stats.norm.pdf(xs), "k--", lw=1.5, label="N(0,1)")
plt.legend(); plt.show()
print(f"empirical P(|T|>2) = {np.mean(np.abs(T)>2):.4f}")
print(f"theory  P(|t_5|>2) = {2*stats.t.sf(2, df=n-1):.4f}")
print(f"theory  P(|Z|>2)   = {2*stats.norm.sf(2):.4f}")
```
The empirical value should be close to the t-theory value and larger than the normal value (heavier tails).

**4.**

```python
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
rng = np.random.default_rng(2)
d1, d2 = 4, 12
n_sim = 100_000
Q1 = (rng.standard_normal(size=(n_sim, d1))**2).sum(axis=1)
Q2 = (rng.standard_normal(size=(n_sim, d2))**2).sum(axis=1)
F = (Q1 / d1) / (Q2 / d2)
xs = np.linspace(1e-3, F.max()*0.9, 400)
plt.hist(F, bins=120, density=True, alpha=0.6)
plt.plot(xs, stats.f.pdf(xs, dfn=d1, dfd=d2), "r-", lw=2)
plt.show()
print(f"empirical mean = {F.mean():.4f}  (theory {d2/(d2-2):.4f})")
```
The empirical mean should be close to \(12/10 = 1.2\).

**5.** The t distribution has heavier tails because the denominator \(\sqrt{Q/\nu}\) is random: small values of \(Q\) inflate \(|T|\), producing more mass in the tails than a fixed-\(\sigma\) normal would. As \(\nu \to \infty\), \(Q/\nu \to 1\) (LLN), so the denominator converges to 1 and \(T \to Z \sim N(0,1)\).

```python
from scipy import stats
for nu in [3, 10, 50, 1000]:
    print(f"nu={nu:5d}: P(|t_nu|>3) = {2*stats.t.sf(3, df=nu):.6f}")
print(f"N(0,1) : P(|Z|>3)   = {2*stats.norm.sf(3):.6f}")
```
At \(\nu=3\) the tail probability is much larger than the normal value; by \(\nu=1000\) it is essentially equal to the normal value, confirming the convergence.

</details>
